<div style="
  background: linear-gradient(145deg, #1a0b08, #2d1310);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #fff8f6;
  box-shadow: 0 6px 14px rgba(0,0,0,0.3);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #ff7b00, #ff0054, #9d0208);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>09 $\rightarrow$ Comprehensive Guide to LLM Knowledge Benchmarks</b>
  <br>
  <span style="color:#ffb5a7; font-size: 18px;">(Evolution and Evaluation)</span>
</div>




---

# Table of Contents

1. [Introduction to Knowledge Benchmarking](#1-introduction-to-knowledge-benchmarking)
2. [MMLU (Massive Multitask Language Understanding)](#2-mmlu-massive-multitask-language-understanding)
3. [TruthfulQA](#3-truthfulqa)
4. [AGIEval](#4-agieval)
5. [GPQA (Google-Proof Q&A)](#5-gpqa-google-proof-qa)
6. [MMLU-Pro](#6-mmlu-pro)
7. [SimpleQA](#7-simpleqa)
8. [HLE (Humanities' Last Exam)](#8-hle-humanities-last-exam)

---




# Learning Objectives

After completing this documentation, the learner will be able to:

- Understand how parametric knowledge is systematically evaluated in Large Language Models.
- Trace the architectural evolution of benchmarking from broad factual testing to frontier-level research exams.
- Differentiate between breadth, depth, and calibration in model evaluation.
- Programmatically implement extraction and evaluation logic for standard benchmarks.
- Identify core limitations in model evaluation, including data contamination, label errors, and LLM-as-a-judge drift.
- Formulate configuration strategies for rigorous and reproducible model benchmarking.

---




# Prerequisites

- Foundational knowledge of Large Language Models (LLMs) and Transformer architectures.
- Understanding of prompt engineering techniques (Zero-shot, Few-shot, Chain-of-Thought).
- Basic familiarity with model hyperparameters (Temperature, Top-P).
- Understanding of probability distributions (Log Likelihoods) in generative AI.

---




<a id="1-introduction-to-knowledge-benchmarking"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">1. Introduction to Knowledge Benchmarking</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Knowledge capability dictates how much factual world knowledge an LLM retains within its weights (parametric knowledge) during the pre-training process. Systematically evaluating this knowledge requires rigorous, standardized datasets rather than naive, localized querying.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Parametric Knowledge:** The internal information an LLM absorbs and stores in its weights and biases from its massive training corpus.
- **Systematic Evaluation:** Moving beyond anecdotal testing to standardized datasets that provide empirical accuracy scores.
- **Benchmark Saturation:** A phenomenon where frontier models consistently score near the upper limit (e.g., >90%), rendering the benchmark obsolete for distinguishing between newer models.
- **Data Contamination:** The leakage of public benchmark test sets into the training data of subsequent model generations, leading to artificial performance inflation (memorization rather than understanding).

#### The Evolutionary Roadmap of Knowledge Benchmarks

```text
                        [ MMLU (2020) ]
                        Breadth Focus
                               |
       -------------------------------------------------
       |             |                 |               |
[ TruthfulQA ]  [ AGIEval ]        [ GPQA ]      [ MMLU-Pro ]
  (2021)          (2023)            (2023)          (2024)
Reliability     Human Exams      Extreme Depth   MMLU Repaired
       |                               |
[ SimpleQA ]                    [ HLE (2025) ]
  (2024)                    Breadth x Extreme Depth
Calibration                       Multimodal
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [1]:
# Standard benchmark execution environment dictionary
evaluation_config = {
    "dataset_name": str,
    "num_few_shot": int,
    "temperature": float,
    "pass_at_k": int,
    "allow_chain_of_thought": bool
}


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- `dataset_name`: The specific benchmark suite being executed.
- `num_few_shot`: The number of solved examples provided in the context window.
- `temperature`: The randomness parameter for generation.
- `pass_at_k`: The number of generations evaluated to find one correct answer.
- `allow_chain_of_thought`: Boolean indicating if reasoning traces are permitted.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `accuracy_score`: A float representing the overall correct predictions divided by the total dataset size.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [2]:
def configure_benchmark_run(benchmark_type):
    """Generates a standardized configuration for knowledge benchmarks."""
    config = {
        "temperature": 0.0,
        "pass_at_k": 1,
        "allow_tools": False,
        "allow_chain_of_thought": False
    }
    
    if benchmark_type == "MMLU":
        config["num_few_shot"] = 5
    elif benchmark_type == "TruthfulQA":
        config["num_few_shot"] = 0
        
    return config

print(configure_benchmark_run("MMLU"))


{'temperature': 0.0, 'pass_at_k': 1, 'allow_tools': False, 'allow_chain_of_thought': False, 'num_few_shot': 5}


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- The function initializes a highly deterministic configuration (`temperature=0.0`).
- Models are restricted from using external tools (`allow_tools=False`) to ensure pure parametric knowledge is tested.
- Specific benchmarks mutate the configuration; MMLU strictly uses a 5-shot prompt, whereas TruthfulQA relies on a 0-shot approach.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
{'temperature': 0.0, 'pass_at_k': 1, 'allow_tools': False, 'allow_chain_of_thought': False, 'num_few_shot': 5}
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Always evaluate models using `temperature=0.0` for highly deterministic and reproducible benchmarking.
- Ensure external tools (web search, compilers) are disabled during pure knowledge testing.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- Testing different models using varying prompt structures. Prompt format sensitivity is extremely high in knowledge benchmarks.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- **Warning:** Higher capabilities do not automatically guarantee higher alignment.
- Early general models often hallucinate aggressively when answering questions outside their actual retained knowledge.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

Benchmarking is the empirical backbone of LLM evaluation. The field constantly evolves as models saturate existing benchmarks, forcing the creation of more difficult, complex, and specialized testing frameworks.

---




<a id="2-mmlu-massive-multitask-language-understanding"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">2. MMLU (Massive Multitask Language Understanding)</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Introduced in September 2020, MMLU is the foundational benchmark for measuring the breadth of an LLM's knowledge. It evaluates models across an extensive array of disciplines using a standard multiple-choice format.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Dataset Size:** 14,000 multiple-choice questions.
- **Subject Diversity:** 57 subjects categorized into four primary domains (STEM, Humanities, Social Sciences, Others).
- **Evaluation Mechanism:** Tests breadth of knowledge rather than extreme depth.
- **Saturation Timeline:** Initial models (like GPT-3) scored ~43%. By 2024, frontier models clustered around 86-90%, reaching near saturation.
- **Label Errors:** Approximately 6.5% of MMLU questions contain flawed or missing correct answers, creating a hard ceiling on achievable accuracy.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [3]:
def evaluate_mmlu(model_output_probs, target_options):
    """Placeholder function signature for MMLU probability-based answer selection."""
    pass


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- `model_output_probs`: A dictionary mapping vocabulary tokens to their calculated log probabilities.
- `target_options`: A list of valid selection tokens (e.g., `["A", "B", "C", "D"]`).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `predicted_answer`: The string token representing the model's selected answer.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [4]:
import numpy as np

def extract_mmlu_answer(log_probs):
    """
    Extracts the multiple choice answer by finding the token 
    with the highest assigned log likelihood.
    """
    valid_options = ["A", "B", "C", "D"]
    
    # Filter logits down to only the valid option characters
    option_probs = {opt: log_probs.get(opt, -np.inf) for opt in valid_options}
    
    # Return the option with the maximum probability
    predicted_answer = max(option_probs, key=option_probs.get)
    return predicted_answer

# Simulated model logits for a specific question
simulated_log_probs = {"A": -2.3, "B": -0.5, "C": -4.1, "D": -5.2, "The": -1.1}
print(f"Model Prediction: {extract_mmlu_answer(simulated_log_probs)}")


Model Prediction: B


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- Instead of relying on the model to generate a text string (which can be unstructured), the standard practice for MMLU is to evaluate the underlying probability distribution.
- The `log_probs` dictionary represents the model's confidence across the vocabulary.
- The logic strictly isolates tokens `A`, `B`, `C`, and `D`, ignoring the rest of the vocabulary.
- The token with the highest probability (closest to zero) is declared the model's answer.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
Model Prediction: B
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Utilize Log Probability evaluation over raw text generation. Text generation requires an "LLM-as-a-judge" to parse the response, introducing extraction errors. Log-likelihoods are mathematically objective.
- Maintain a strict 5-shot prompt configuration to ensure parity with historical MMLU testing.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- Comparing generation-based accuracy scores with log-likelihood accuracy scores. They often yield a 1-3% variance for the exact same model.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- **Limitation:** MMLU tests fact recall but does not evaluate reasoning traces, open-ended retrieval, or calibration.
- The dataset is English-only and heavily skewed toward Western curricula.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

MMLU established the standard for breadth testing in LLMs. However, due to its static nature, high contamination rates, and inherent label errors, it is considered saturated as of 2024.

---




<a id="3-truthfulqa"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">3. TruthfulQA</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Launched in September 2021, TruthfulQA tests a model's reliability by forcing it to confront common human misconceptions and internet myths, measuring truthfulness against scale.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Dataset Size:** 817 adversarial questions across 38 categories.
- **Inverse Scaling Law Anomaly:** Early testing revealed that larger, more capable models performed *worse* on this benchmark. Because larger models absorbed more internet data, they absorbed more widespread falsehoods (e.g., the myth that "cracking knuckles causes arthritis").
- **Focus:** Evaluates Alignment and Truthfulness over pure capability.
- **Contamination Source:** Unlike other benchmarks contaminated during pre-training, TruthfulQA data often leaks during the alignment phase (RLHF/Instruction Tuning).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [5]:
def calculate_mc2_score(log_probs, true_answer_indices):
    """Placeholder signature for TruthfulQA multi-true normalized scoring."""
    pass


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- `log_probs`: Dictionary mapping option indices to probability values.
- `true_answer_indices`: List of indices representing all factually correct answers for a given question.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `mc2_score`: Float representing the normalized probability mass assigned to true statements.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [6]:
def calculate_mc2(log_probs, true_indices):
    """
    Implements the MC2 scoring methodology, normalizing the sum of probabilities 
    assigned to all correct answers.
    """
    total_mass = sum(log_probs.values())
    
    # Sum probabilities of only the correct options
    true_mass = sum(log_probs[idx] for idx in true_indices if idx in log_probs)
    
    # Calculate normalized probability mass
    return true_mass / total_mass

# Question: What happens if you crack your knuckles?
# Options: 0 (Arthritis - False), 1 (Sound - True), 2 (Nothing - True)
probs = {0: 0.60, 1: 0.25, 2: 0.15}
true_answers = [1, 2]

print(f"MC2 Score: {calculate_mc2(probs, true_answers):.2f}")


MC2 Score: 0.40


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- TruthfulQA questions frequently contain multiple valid true answers and multiple misconceptions.
- The `MC2` (Multi-True) metric sums the probability mass assigned to *all* correct answers.
- The result is divided by the total probability mass to yield a normalized score, ensuring a comprehensive view of the model's internal belief distribution.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
MC2 Score: 0.40
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Utilize the `MC2` metric as the primary reporting standard for TruthfulQA, as it accurately reflects questions with multiple correct options.
- Append exactly 6 fixed, unrelated QA examples to the system prompt (simulated zero-shot).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- Using the benchmark to evaluate factual recall. TruthfulQA is strictly an adversarial test of misconceptions, not a general knowledge evaluation.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- TruthfulQA scores drastically improved globally once alignment techniques like RLHF (Reinforcement Learning from Human Feedback) and DPO (Direct Preference Optimization) matured.
- **Limitation:** It does not test honesty under pressure (whether a model will lie if prompted aggressively).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

TruthfulQA proved that raw scale and capability do not inherently result in safe or aligned models, kickstarting the industry's intense focus on post-training alignment techniques.

---




<a id="4-agieval"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">4. AGIEval</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Released in April 2023, AGIEval shifted the benchmarking paradigm by utilizing standard human examinations to directly compare LLM capabilities against human baselines.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Dataset Size:** 8,000+ questions spanning 20 exam sections.
- **Sources:** Real-world standardized tests including the SAT, LSAT, Civil Service exams, and the Chinese Gaokao.
- **Bilingual Structure:** The first major benchmark split evenly between English and Chinese logic/knowledge.
- **Human Baseline:** Measured directly against actual human test-takers (Average Human: ~67%, Top Human: ~91%). Initial frontier models (GPT-4) scored ~58%.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [7]:
def formulate_agieval_prompt(question_text, language="EN"):
    """Placeholder signature for bilingual AGIEval prompt formulation."""
    pass


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- `question_text`: The raw text of the exam question.
- `language`: Enum/String specifying the language routing (e.g., "EN", "ZH").

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `formatted_prompt`: A structured string prompting the model appropriately based on the language.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [8]:
def generate_bilingual_prompt(question, lang="EN"):
    """
    Formats prompts dynamically based on target language routing for AGIEval.
    Supports English (EN) and Chinese (ZH) evaluation subsets.
    """
    templates = {
        "EN": "Please answer the following multiple-choice question and provide the final answer:
",
        "ZH": "Please answer the following standardized exam question (Chinese subset):
"
    }
    
    prefix = templates.get(lang.upper(), templates["EN"])
    return f"{prefix}{question}"

q_en = "If x + 2 = 5, what is x?"
q_zh = "If x + 2 = 5, find the value of x. (Gaokao Mathematics Subset)"

print("English Prompt:")
print(generate_bilingual_prompt(q_en, "EN"))
print("
Chinese Subset Prompt:")
print(generate_bilingual_prompt(q_zh, "ZH"))


SyntaxError: unterminated string literal (detected at line 7) (2434373943.py, line 7)

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- The dataset routes questions natively through language-specific instruction templates.
- Models are forced to process not just parametric knowledge, but semantic understanding across distinct linguistic frameworks simultaneously.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
English Prompt:
Please answer the following multiple-choice question and provide the final answer:
If x + 2 = 5, what is x?

Chinese Subset Prompt:
Please answer the following standardized exam question (Chinese subset):
If x + 2 = 5, find the value of x. (Gaokao Mathematics Subset)
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Enable Chain-of-Thought (CoT) prompting. Standardized exams require multi-step reasoning, making zero-shot direct answers less effective.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- **Marketing Fallacy:** Equating an LLM beating a human exam score to achieving Artificial General Intelligence (AGI). Passing an exam demonstrates pattern matching and knowledge retrieval, not long-horizon agentic task completion or reasoning in novel environments.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- Mixed Format: While mostly MCQs, two exams within the dataset require short-answer generation.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

AGIEval provided the first robust, quantifiable anchor mapping LLM performance directly to established human cognitive baselines across multiple languages.

---




<a id="5-gpqa-google-proof-qa"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">5. GPQA (Google-Proof Q&A)</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Introduced in November 2023, GPQA tests the extreme depth of an LLM's knowledge in highly specialized scientific domains. It is specifically designed to be "Google-Proof."

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Domains:** Strictly restricted to Physics, Chemistry, and Biology.
- **Difficulty:** PhD-level questions. A non-expert human provided with unrestricted internet access for 30 minutes cannot solve a single question.
- **Expert Validation:** Every question was written and strictly cross-validated by two domain experts.
- **Subsets:**
  - **Main:** 443 questions.
  - **Extended:** 546 questions.
  - **Diamond:** 198 ultra-hard, rigorously vetted questions (the primary reporting standard).
- **Scores:** Early GPT-4 scored ~39%. Specialized reasoning models later pushed this to ~78-87%.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [ ]:
def configure_gpqa_run():
    """Placeholder signature for GPQA Diamond setup configuration."""
    pass


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- None

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `config`: Dictionary with GPQA specific execution constraints.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [ ]:
def setup_gpqa_environment():
    """Sets up the strict evaluation environment for GPQA Diamond."""
    return {
        "dataset": "GPQA_Diamond",
        "shot_count": 0,
        "temperature": 0.0,
        "allow_chain_of_thought": True, # Crucial for PhD level reasoning
        "allow_tools": False
    }

print(setup_gpqa_environment())


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- GPQA relies heavily on the model's internal reasoning engine. Therefore, `allow_chain_of_thought` is enabled.
- External tools are strictly disabled. The benchmark tests if the model has internalized deep scientific logic.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
{'dataset': 'GPQA_Diamond', 'shot_count': 0, 'temperature': 0.0, 'allow_chain_of_thought': True, 'allow_tools': False}
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Always report performance explicitly against the **Diamond** subset when publishing GPQA results, as it contains the highest quality, error-free questions.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- Assuming GPQA measures general graduate knowledge. It strictly measures advanced STEM capability.
- Trusting the final answer as proof of correct logic. GPQA only evaluates the final generated character, not the validity of the internal reasoning trace (models can guess correctly using flawed logic).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- **Limitation:** The dataset is statistically extremely small (198 questions in Diamond), resulting in wide confidence intervals and potential reliability issues.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

GPQA shifted the evaluation landscape from broad, shallow factual recall to extreme, vertical depth, proving highly beneficial for evaluating reasoning-heavy models (like OpenAI's o1 series).

---




<a id="6-mmlu-pro"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">6. MMLU-Pro</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Released in 2024, MMLU-Pro is a direct architectural upgrade to the original MMLU, engineered specifically to repair its fundamental flaws, eliminate noise, and significantly increase difficulty.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Dataset Size:** 12,000 questions.
- **Discipline Consolidation:** Reduced from 57 granular subjects to 14 broad categories to prevent under-representation of core subjects.
- **Structural Upgrade:** Expanded from 4 options per question to **10 options** per question.
- **Reasoning Shift:** Trivial fact-retrieval questions were purged and replaced with complex, calculation and reasoning-based questions.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [ ]:
def evaluate_mmlu_pro(logits, options_count=10):
    """Placeholder signature for MMLU-Pro 10-option MCQ parsing."""
    pass


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- `logits`: Output probabilities from the model.
- `options_count`: Integer representing the number of valid MCQ options.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `predicted_answer`: The calculated selection.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [ ]:
import string

def validate_mmlu_pro_schema():
    """Generates the valid option schema for MMLU-Pro."""
    # MMLU-Pro uses options A through J (10 options)
    valid_options = list(string.ascii_uppercase[:10])
    return valid_options

print(f"MMLU-Pro Valid Options: {validate_mmlu_pro_schema()}")


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- Standard MMLU logic filters logits down to `[A, B, C, D]`. MMLU-Pro requires the evaluation pipeline to support a heavily expanded logical search space `[A, B, C, D, E, F, G, H, I, J]`.
- This simple architectural change massively reduces the model's ability to guess or arrive at the answer via simple elimination.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
MMLU-Pro Valid Options: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Standardize testing with a 5-shot prompt and explicitly invoke Chain-of-Thought. Reasoning models typically score ~20 points higher on MMLU-Pro compared to standard generation models.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- Reusing legacy MMLU parsing scripts. If the evaluation harness is not updated to capture options up to `J`, the pipeline will crash or drop answers.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- **Limitation:** MMLU-Pro lacks a defined human baseline.
- Because it pulls heavily from public STEM problem sets, it suffers from a high risk of source contamination.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

MMLU-Pro effectively resuscitated the MMLU format by increasing structural complexity. It heavily favors models possessing strong internal reasoning capabilities over pure memorization.

---




<a id="7-simpleqa"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">7. SimpleQA</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Launched in 2024, SimpleQA is a unique benchmark focusing on short-form factual generation and model calibration (the model's awareness of its own ignorance).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Dataset Size:** 4,326 short, fact-seeking questions.
- **Selection Bias:** Composed entirely of questions that GPT-4 specifically failed to answer.
- **Format:** Pure text generation (No multiple choice).
- **Calibration Testing:** Models are encouraged to output a refusal (e.g., "I don't know" or "Pass") if they lack confidence, penalizing uncalibrated hallucinations.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [ ]:
def calculate_simpleqa_metrics(total_q, attempted_q, correct_q):
    """Placeholder signature for SimpleQA calibration & F-score calculation."""
    pass


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- `total_q`: Total questions in the dataset.
- `attempted_q`: Number of questions the model actually attempted to answer (did not output "pass").
- `correct_q`: Number of accurate answers generated.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `tuple`: (Overall Accuracy, Attempted Accuracy, F-Score).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [ ]:
def calculate_simpleqa_fscore(total, attempted, correct):
    """
    Calculates the harmonic mean (F-Score) between overall correctness 
    and calibrated correctness.
    """
    # Metric 1: Correctness across the entire board
    accuracy_overall = correct / total
    
    # Metric 2: Correctness ONLY on questions the model felt confident answering
    accuracy_attempted = correct / attempted if attempted > 0 else 0
    
    if accuracy_overall + accuracy_attempted == 0:
        return 0.0, 0.0, 0.0
        
    # Metric 3: F-Score balances factuality with humility
    f_score = 2 * (accuracy_overall * accuracy_attempted) / (accuracy_overall + accuracy_attempted)
    
    return accuracy_overall, accuracy_attempted, f_score

metrics = calculate_simpleqa_fscore(total=4326, attempted=3500, correct=2100)
print(f"Overall Acc: {metrics[0]:.2f} | Attempted Acc: {metrics[1]:.2f} | F-Score: {metrics[2]:.2f}")


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- The scoring mechanism separates pure accuracy from calibrated accuracy.
- If a model attempts every question and hallucinates wildly, its `accuracy_attempted` crashes, destroying the F-Score.
- If a model selectively answers only what it knows, `accuracy_attempted` remains high, balancing the F-score and proving high model calibration.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
Overall Acc: 0.49 | Attempted Acc: 0.60 | F-Score: 0.54
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Utilize an updated, highly capable LLM (e.g., GPT-4o) as the grader/judge. Since answers are open text, programmatic regex matching is insufficient.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- Comparing historical SimpleQA scores directly with modern runs. Because the evaluation relies on an "LLM-as-a-judge," the judge's capabilities drift and improve over time, altering baseline strictness.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- **Staleness:** Factual answers (e.g., "Who is the Rank 1 Rugby Player?") decay over time, requiring dataset maintenance.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

SimpleQA is an active, highly difficult benchmark that proves open-ended factual generation paired with calibration checking remains a massive hurdle for modern AI.

---




<a id="8-hle-humanities-last-exam"></a>
##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">8. HLE (Humanities' Last Exam)</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Overview</span>

Introduced in January 2025, HLE represents the ultimate threshold of closed-ended knowledge evaluation. It merges unprecedented breadth with extreme depth, acting as a final frontier benchmark before evaluation transitions entirely to open-ended agentic tasks.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Core Concepts</span>

- **Dataset Size:** 2,500 questions spanning over 100 highly specialized subjects.
- **Authorship:** Crafted by over 1,000 domain experts globally.
- **Format Architecture:**
  - 80% Short Answer Generation.
  - 20% Multiple Choice.
  - 10% Multimodal (incorporates visual/image inputs).
- **Confidence Calibration:** Models must explicitly output a percentage representing their confidence in their generated answer.
- **Withheld Private Set:** The research group maintains a fully private test set off the internet to permanently safeguard against pre-training data contamination.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Syntax</span>




In [ ]:
def evaluate_hle_calibration(predictions, actuals, confidence_scores):
    """Placeholder signature for HLE Root Mean Square Error (RMSE) calibration calculation."""
    pass


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Parameters</span>

- `predictions`: List of model-generated answers.
- `actuals`: List of ground truth answers.
- `confidence_scores`: List of float values (0.0 to 1.0) representing the model's self-reported confidence.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Return Values</span>

- `rmse`: Float representing the Root Mean Square Error of the model's calibration.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Example</span>




In [ ]:
import math

def calculate_calibration_error(predictions, actuals, confidences):
    """
    Calculates how well a model's self-reported confidence aligns with 
    its actual correctness using Root Mean Square Error (RMSE).
    """
    squared_errors = []
    
    for pred, actual, conf in zip(predictions, actuals, confidences):
        # 1.0 if correct, 0.0 if incorrect
        is_correct = 1.0 if pred == actual else 0.0
        
        # Error is the difference between confidence and actual correctness
        error = conf - is_correct
        squared_errors.append(error ** 2)
        
    mean_squared_error = sum(squared_errors) / len(squared_errors)
    rmse = math.sqrt(mean_squared_error)
    
    return rmse

preds = ["A", "B", "C"]
truths = ["A", "C", "C"] # 2nd prediction is wrong
confs = [0.9, 0.8, 0.6]  # Confident about the wrong answer

print(f"Calibration RMSE: {calculate_calibration_error(preds, truths, confs):.3f}")


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Code Walkthrough</span>

- The calibration engine maps the model's text generation directly against its internal confidence metric.
- If a model is highly confident (`0.8`) but wrong (`0.0`), the squared error is large.
- The RMSE metric heavily penalizes models that exhibit arrogant hallucinations.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Expected Output</span>

```text
Calibration RMSE: 0.516
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Best Practices</span>

- Ensure testing infrastructure supports multimodal (image) inputs, as models lacking vision capabilities automatically forfeit ~10% of the maximum possible score.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Common Mistakes</span>

- Assuming HLE represents everyday general knowledge. It is exclusively compiled from frontier failure cases (questions that stumped models in 2024/2025).

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Important Notes</span>

- HLE is currently highly active. If frontier models achieve saturation on HLE, the industry consensus is that closed-ended QA benchmarking will be retired entirely in favor of long-horizon agent evaluation.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">Key Takeaways</span>

HLE is the pinnacle of static knowledge benchmarks, combining vast breadth, PhD-level depth, multimodality, and strict calibration testing, protected by a withheld private test set.

---


